## Importing Libraries

In [981]:
import mysql.connector
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

## Connecting to Database

In [982]:
# Database connection details
host = "srv610.hstgr.io"
user = "u385006994_freelance"
password = "FredbAc3s5"

try:
    conn = mysql.connector.connect(
        host=host,
        user=user,
        password=password
    )
    cursor = conn.cursor()

    # List all databases
    cursor.execute("SHOW DATABASES;")
    databases = cursor.fetchall()
    
    print("Tables in database:")
    for db in databases:
        print(db[0])

except mysql.connector.Error as err:
    print(f"Error: {err}")

finally:
    if 'conn' in locals() and conn.is_connected():
        cursor.close()
        conn.close()
        print("Database connection closed.")


Tables in database:
information_schema
u385006994_databoks
Database connection closed.


In [983]:
database_name = "u385006994_databoks"  # Actual database name

try:
    conn = mysql.connector.connect(
        host=host,
        user=user,
        password=password,
        database=database_name
    )
    cursor = conn.cursor()

    # List all tables in the selected database
    cursor.execute("SHOW TABLES;")
    tables = cursor.fetchall()
    
    print(f"Tables in '{database_name}':")
    for table in tables:
        print(table[0])

except mysql.connector.Error as err:
    print(f"Error: {err}")

finally:
    if 'conn' in locals() and conn.is_connected():
        cursor.close()
        conn.close()
        print("Database connection closed.")


Tables in 'u385006994_databoks':
cuaca_kabkota_new
data
Database connection closed.


In [984]:
table_name = "data"  # Replace with an actual table name

try:
    conn = mysql.connector.connect(
        host=host,
        user=user,
        password=password,
        database=database_name
    )
    cursor = conn.cursor()

    # Get column names
    cursor.execute(f"DESCRIBE {table_name};")
    columns = [col[0] for col in cursor.fetchall()]
    
    print(f"Columns in '{table_name}': {columns}")

    # Fetch first 5 rows
    cursor.execute(f"SELECT * FROM {table_name} LIMIT 5;")
    rows = cursor.fetchall()

    # Convert to DataFrame
    df = pd.DataFrame(rows, columns=columns)
    display(df)

except mysql.connector.Error as err:
    print(f"Error: {err}")

finally:
    if 'conn' in locals() and conn.is_connected():
        cursor.close()
        conn.close()
        print("Database connection closed.")

Columns in 'data': ['id', 'id_nama_data', 'indikator', 'satuan', 'data_x', 'data_y', 'namadata_tsidx', 'date_created']


,id,id_nama_data,indikator,satuan,data_x,data_y,namadata_tsidx,date_created
0,329339,11164,Cadangan Devisa BI Menurut Jenis,US$ juta,"31-01-2001,28-02-2001,31-03-2001,30-04-2001,31...","29257.0,29107.0,28673.0,28713.0,28594.0,28638....",Cadangan Devisa BI Menurut Jenis Total Indones...,2025-01-30 05:33:40
1,181121,8453,Total Nilai Ekspor Bulanan,US$,"31-01-1993,28-02-1993,31-03-1993,30-04-1993,31...","3001900000.0,2892500000.0,3008500000.0,2957500...",Total Nilai Ekspor Impor Bulanan Ekspor Indone...,2025-01-30 05:50:12
2,181124,8453,Total Nilai Impor Bulanan,US$,"31-01-1993,28-02-1993,31-03-1993,30-04-1993,31...","2143100000.0,2015900000.0,2191200000.0,2234800...",Total Nilai Ekspor Impor Bulanan Impor Indones...,2025-01-30 05:50:12
3,183171,9350,Pertumbuhan PDB Harga Konstan Menurut Lapangan...,persen,"31-03-2011,30-06-2011,30-09-2011,31-12-2011,31...","6.48,6.27,6.01,5.94,6.11,6.21,5.94,5.87,5.54,5...",Laju Pertumbuhan PDB Harga Konstan (Triwulanan...,2025-01-30 05:56:34


Database connection closed.


In [985]:
csv_filename = "exported_data.csv"

df.to_csv(csv_filename, index=False)
print(f"Dataset has been successfully exported to '{csv_filename}'")

Dataset has been successfully exported to 'exported_data.csv'


## Data Transformation

In [986]:
def transform_long_format(df):
    long_format_data = []
    
    for _, row in df.iterrows():
        dates = row['data_x'].split(',')
        values = row['data_y'].split(',')
        
        if len(dates) == len(values):
            for date, value in zip(dates, values):
                long_format_data.append({
                    'id': row['id'],
                    'id_nama_data': row['id_nama_data'],
                    'indikator': row['indikator'],
                    'satuan': row['satuan'],
                    'date': date,
                    'value': float(value) if value else None,  # Convert to float
                    'namadata_tsidx': row['namadata_tsidx']
                })

    return pd.DataFrame(long_format_data)

# Apply transformation
df_long = transform_long_format(df)

# Convert 'date' column to datetime format
df_long['date'] = pd.to_datetime(df_long['date'], errors='coerce')
df_long.head()

/var/folders/0t/tym83fts64b5fv862t1vq5lm0000gn/T/ipykernel_89758/3505989653.py:26: UserWarning:

Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.



,id,id_nama_data,indikator,satuan,date,value,namadata_tsidx
0,329339,11164,Cadangan Devisa BI Menurut Jenis,US$ juta,2001-01-31,29257.00000000,Cadangan Devisa BI Menurut Jenis Total Indones...
1,329339,11164,Cadangan Devisa BI Menurut Jenis,US$ juta,2001-02-28,29107.00000000,Cadangan Devisa BI Menurut Jenis Total Indones...
2,329339,11164,Cadangan Devisa BI Menurut Jenis,US$ juta,2001-03-31,28673.00000000,Cadangan Devisa BI Menurut Jenis Total Indones...
3,329339,11164,Cadangan Devisa BI Menurut Jenis,US$ juta,2001-04-30,28713.00000000,Cadangan Devisa BI Menurut Jenis Total Indones...
4,329339,11164,Cadangan Devisa BI Menurut Jenis,US$ juta,2001-05-31,28594.00000000,Cadangan Devisa BI Menurut Jenis Total Indones...


In [987]:
# Only records from 2015 to 2024
df_long = df_long[(df_long['date'].dt.year >= 2000) & (df_long['date'].dt.year <= 2024)]
df_long.head()

,id,id_nama_data,indikator,satuan,date,value,namadata_tsidx
0,329339,11164,Cadangan Devisa BI Menurut Jenis,US$ juta,2001-01-31,29257.00000000,Cadangan Devisa BI Menurut Jenis Total Indones...
1,329339,11164,Cadangan Devisa BI Menurut Jenis,US$ juta,2001-02-28,29107.00000000,Cadangan Devisa BI Menurut Jenis Total Indones...
2,329339,11164,Cadangan Devisa BI Menurut Jenis,US$ juta,2001-03-31,28673.00000000,Cadangan Devisa BI Menurut Jenis Total Indones...
3,329339,11164,Cadangan Devisa BI Menurut Jenis,US$ juta,2001-04-30,28713.00000000,Cadangan Devisa BI Menurut Jenis Total Indones...
4,329339,11164,Cadangan Devisa BI Menurut Jenis,US$ juta,2001-05-31,28594.00000000,Cadangan Devisa BI Menurut Jenis Total Indones...


In [988]:
missing_values = df_long.isnull().sum()
missing_values

id                0
id_nama_data      0
indikator         0
satuan            0
date              0
value             0
namadata_tsidx    0
dtype: int64

In [989]:
csv_filename = "transformed_data.csv"
df_long.to_csv(csv_filename, index=False)

print(f"Transformed dataset has been successfully exported to '{csv_filename}'")

Transformed dataset has been successfully exported to 'transformed_data.csv'


## Transform to Keep Necessary Data

In [990]:
# Pivoting the DataFrame to create columns for each indicator
df_wide = df_long.pivot(index='date', columns='indikator', values='value')
df_wide = df_wide.reset_index()
df_wide.columns.name = None 

### Forward Filling & Backward Filling Variable Cadangan Devisa

In [991]:
df_wide['Cadangan Devisa BI Menurut Jenis'].fillna(method='ffill', inplace=True)
df_wide['Cadangan Devisa BI Menurut Jenis'].fillna(method='bfill', inplace=True)

/var/folders/0t/tym83fts64b5fv862t1vq5lm0000gn/T/ipykernel_89758/2158547563.py:1: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

/var/folders/0t/tym83fts64b5fv862t1vq5lm0000gn/T/ipykernel_89758/2158547563.py:2: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



### Forward Filling and Backward Filling Variable Pertumbuhan PDB Harga Konstan Menurut Lapangan Usaha Sektoral (Triwulanan YoY)

In [992]:
# Forward Filling
df_wide['Pertumbuhan PDB Harga Konstan Menurut Lapangan Usaha Sektoral (Triwulanan YoY)'].fillna(method='ffill', inplace=True)
df_wide['Pertumbuhan PDB Harga Konstan Menurut Lapangan Usaha Sektoral (Triwulanan YoY)'].fillna(method='bfill', inplace=True)

df_wide.isnull().sum()

/var/folders/0t/tym83fts64b5fv862t1vq5lm0000gn/T/ipykernel_89758/4281799194.py:2: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

/var/folders/0t/tym83fts64b5fv862t1vq5lm0000gn/T/ipykernel_89758/4281799194.py:3: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



date                                                                              0
Cadangan Devisa BI Menurut Jenis                                                  0
Pertumbuhan PDB Harga Konstan Menurut Lapangan Usaha Sektoral (Triwulanan YoY)    0
Total Nilai Ekspor Bulanan                                                        0
Total Nilai Impor Bulanan                                                         0
dtype: int64

In [993]:
df_wide.head()

,date,Cadangan Devisa BI Menurut Jenis,Pertumbuhan PDB Harga Konstan Menurut Lapangan Usaha Sektoral (Triwulanan YoY),Total Nilai Ekspor Bulanan,Total Nilai Impor Bulanan
0,2000-01-31,29257.00000000,6.48000000,4394000000.00000000,2169500000.00000000
1,2000-02-29,29257.00000000,6.48000000,4793800000.00000000,2120400000.00000000
2,2000-03-31,29257.00000000,6.48000000,4951500000.00000000,2265100000.00000000
3,2000-04-30,29257.00000000,6.48000000,5012000000.00000000,2338900000.00000000
4,2000-05-31,29257.00000000,6.48000000,4858000000.00000000,2383600000.00000000


In [994]:
csv_filename = "transformed_data2.csv"
df_wide.to_csv(csv_filename, index=False)

print(f"Transformed dataset has been successfully exported to '{csv_filename}'")

Transformed dataset has been successfully exported to 'transformed_data2.csv'


## Foreign Exchange Currency Rate Dataset

In [995]:
currency_file_path = "/Users/morenomusadat/Proyek Predictive Analytics/investing_idr_/Data Historis IDR_JPY.csv"
df_currency = pd.read_csv(currency_file_path, delimiter=",")
df_currency.columns = df_currency.columns.str.strip()
df_currency.tail()

,Tanggal,Terakhir,Pembukaan,Tertinggi,Terendah,Vol.,Perubahan%
297,01/05/2000,"1,2452","1,3565","1,3790","1,2180",NaN,"-8,80%"
298,01/04/2000,"1,3653","1,3610","1,4108","1,2145",NaN,"0,63%"
299,01/03/2000,"1,3567","1,4829","1,4874","1,3092",NaN,"-8,49%"
300,01/02/2000,"1,4825","1,4376","1,5155","1,3847",NaN,"2,77%"
301,01/01/2000,"1,4425","1,4361","1,4865","1,3549",NaN,"-1,43%"


### Merging Datasets

In [996]:
if 'Tanggal' in df_currency.columns:
    df_currency['date'] = pd.to_datetime(df_currency['Tanggal'], format='%d/%m/%Y', errors='coerce')
    df_currency.drop(columns=['Tanggal'], inplace=True)
elif 'date' in df_currency.columns:
    df_currency['date'] = pd.to_datetime(df_currency['date'], errors='coerce')
else:
    raise KeyError("No date column found in df_currency. Expected 'Tanggal' or 'date'.")

# Keep only the necessary currency variables.
df_currency = df_currency[['date', 'Terakhir', 'Tertinggi', 'Terendah']]

# Convert the currency columns from strings (with commas as decimal separators) into floats.
def parse_currency_value(x):
    return float(str(x).replace(',', '.').strip())

for col in ['Terakhir', 'Tertinggi', 'Terendah']:
    df_currency[col] = df_currency[col].apply(parse_currency_value)

df_currency['month'] = df_currency['date'].dt.strftime('%Y-%m')
df_currency_monthly = df_currency.groupby('month', as_index=False).last()

df_wide['date'] = pd.to_datetime(df_wide['date'], errors='coerce')
df_wide['month'] = df_wide['date'].dt.strftime('%Y-%m')

# Merge the DataFrames
df_merged = pd.merge(
    df_wide,
    df_currency_monthly[['month', 'Terakhir', 'Tertinggi', 'Terendah']],
    on='month',
    how='inner'
)

df_merged.drop('month', axis=1, inplace=True)
print("Merged DataFrame preview:")
df_merged.head()

Merged DataFrame preview:


,date,Cadangan Devisa BI Menurut Jenis,Pertumbuhan PDB Harga Konstan Menurut Lapangan Usaha Sektoral (Triwulanan YoY),Total Nilai Ekspor Bulanan,Total Nilai Impor Bulanan,Terakhir,Tertinggi,Terendah
0,2000-01-31,29257.00000000,6.48000000,4394000000.00000000,2169500000.00000000,1.44250000,1.48650000,1.35490000
1,2000-02-29,29257.00000000,6.48000000,4793800000.00000000,2120400000.00000000,1.48250000,1.51550000,1.38470000
2,2000-03-31,29257.00000000,6.48000000,4951500000.00000000,2265100000.00000000,1.35670000,1.48740000,1.30920000
3,2000-04-30,29257.00000000,6.48000000,5012000000.00000000,2338900000.00000000,1.36530000,1.41080000,1.21450000
4,2000-05-31,29257.00000000,6.48000000,4858000000.00000000,2383600000.00000000,1.24520000,1.37900000,1.21800000


In [997]:
print(df_currency.columns)
print(df_currency.dtypes)

print(df_merged.columns)


df_merged.isnull().sum()

Index(['date', 'Terakhir', 'Tertinggi', 'Terendah', 'month'], dtype='object')
date         datetime64[ns]
Terakhir            float64
Tertinggi           float64
Terendah            float64
month                object
dtype: object
Index(['date', 'Cadangan Devisa BI Menurut Jenis',
       'Pertumbuhan PDB Harga Konstan Menurut Lapangan Usaha Sektoral (Triwulanan YoY)',
       'Total Nilai Ekspor Bulanan', 'Total Nilai Impor Bulanan', 'Terakhir',
       'Tertinggi', 'Terendah'],
      dtype='object')


date                                                                              0
Cadangan Devisa BI Menurut Jenis                                                  0
Pertumbuhan PDB Harga Konstan Menurut Lapangan Usaha Sektoral (Triwulanan YoY)    0
Total Nilai Ekspor Bulanan                                                        0
Total Nilai Impor Bulanan                                                         0
Terakhir                                                                          0
Tertinggi                                                                         0
Terendah                                                                          0
dtype: int64

In [998]:
csv_filename = "fully_merged.csv"
df_merged.to_csv(csv_filename, index=False)

print(f"Transformed dataset has been successfully exported to '{csv_filename}'")

Transformed dataset has been successfully exported to 'fully_merged.csv'


### Final Dataset Visualization

In [999]:
import plotly.express as px
import plotly.graph_objects as go

fig_line = px.line(
    df_merged,
    x='date',
    y=['Terakhir', 'Tertinggi', 'Terendah'],
    title='Currency Variables Over Time',
    labels={
        "date": "Date",
        "value": "Scaled Currency Value",
        "variable": "Currency Variable"
    }
)
fig_line.update_layout(hovermode='x unified')
fig_line.show()

header_values = list(df_merged[['date', 'Terakhir', 'Tertinggi', 'Terendah']].columns)
cell_values = [
    df_merged['date'].dt.strftime('%Y-%m-%d').tolist(),  # formatted date strings
    df_merged['Terakhir'].round(8).tolist(),
    df_merged['Tertinggi'].round(8).tolist(),
    df_merged['Terendah'].round(8).tolist()
]

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=header_values,
        fill_color='paleturquoise',
        align='left'
    ),
    cells=dict(
        values=cell_values,
        fill_color='lavender',
        align='left'
    ))
])

fig_table.update_layout(title='Data Table: Currency Variables')
fig_table.show()


/opt/anaconda3/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



## XGBoost Modeling

### Model Libraries

In [1002]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --------------------------
# Data Preparation
# --------------------------
df = df_merged.copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.sort_values('date').reset_index(drop=True)

# Create a time index: number of months since start.
base_time = df['date'].dt.year.min() * 12 + df['date'].dt.month.min()
df['time_index'] = (df['date'].dt.year * 12 + df['date'].dt.month) - base_time

# Create seasonal features.
df['sin_time'] = np.sin(2 * np.pi * df['time_index'] / 12)
df['cos_time'] = np.cos(2 * np.pi * df['time_index'] / 12)

# Create one-month lag features.
df['Terakhir_lag1']  = df['Terakhir'].shift(1)
df['Tertinggi_lag1'] = df['Tertinggi'].shift(1)
df['Terendah_lag1']  = df['Terendah'].shift(1)
df = df.dropna().reset_index(drop=True)

# Multiply target values by 1e5 (for numerical stability; will reverse later).
targets = ['Terakhir', 'Tertinggi', 'Terendah']
df[targets] = df[targets] * 1e5

# Define features.
features = [
    'Cadangan Devisa BI Menurut Jenis',
    'Pertumbuhan PDB Harga Konstan Menurut Lapangan Usaha Sektoral (Triwulanan YoY)',
    'Total Nilai Ekspor Bulanan',
    'Total Nilai Impor Bulanan',
    'time_index',
    'sin_time',
    'cos_time',
    'Terakhir_lag1',
    'Tertinggi_lag1',
    'Terendah_lag1'
]

# --------------------------
# Data Splitting
# --------------------------
df['month_period'] = df['date'].dt.to_period('M')
unique_months = sorted(df['month_period'].unique())
if len(unique_months) < 3:
    raise ValueError("Not enough data for splitting (need at least 3 unique months).")
# Reserve the last 2 months for validation.
val_months = unique_months[-2:]
validation = df[df['month_period'].isin(val_months)].copy()
train_test = df[~df['month_period'].isin(val_months)].copy()
df.drop('month_period', axis=1, inplace=True)

# Split train_test into training (80%) and testing (10%).
n_tt = len(train_test)
train_end = int(n_tt * 0.8)
test_end  = int(n_tt * 0.9)
train_data = train_test.iloc[:train_end].copy()
test_data  = train_test.iloc[train_end:test_end].copy()

X_train = train_data[features]
y_train = train_data[targets]
X_test  = test_data[features]
y_test  = test_data[targets]

# For forecasting, define the exogenous feature list (the original 4 exogenous variables).
exog_features = [
    'Cadangan Devisa BI Menurut Jenis',
    'Pertumbuhan PDB Harga Konstan Menurut Lapangan Usaha Sektoral (Triwulanan YoY)',
    'Total Nilai Ekspor Bulanan',
    'Total Nilai Impor Bulanan'
]

# --------------------------
# Feature Scaling
# --------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# --------------------------
# Model Training
# --------------------------
model = MultiOutputRegressor(
    xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
)
model.fit(X_train_scaled, y_train)
print("Test MSE:", mean_squared_error(y_test, model.predict(X_test_scaled)))

# --------------------------
# Iterative Forecasting (2 Years Ahead)
# --------------------------
# To ensure the forecast line connects with historical data, we prepend the last historical observation.
last_hist_date = train_test['date'].max()
# For continuity, set the first forecast date equal to the last historical observation.
forecast_start_date = last_hist_date  
# Generate 25 month-end dates (first point is the anchor, then 24 future forecasts).
full_horizon = pd.date_range(start=forecast_start_date, periods=25, freq='M')

# Exogenous inputs: if forecast date exists in validation, use its values; otherwise use constant exogenous values from the last observation.
last_exog = train_test.iloc[-1][exog_features].astype(float).values
anchor_exog = last_exog.copy()
anchor = np.array([
    train_test.iloc[-1]['Terakhir'],
    train_test.iloc[-1]['Tertinggi'],
    train_test.iloc[-1]['Terendah']
])
last_obs = train_test.iloc[-1].copy()

forecast_results = []
for i, forecast_date in enumerate(full_horizon):
    if i == 0:
        raw_pred = anchor.copy()
    else:
        forecast_time_index = (forecast_date.year * 12 + forecast_date.month) - base_time
        sin_time = np.sin(2 * np.pi * forecast_time_index / 12)
        cos_time = np.cos(2 * np.pi * forecast_time_index / 12)
        
        row_val = validation[validation['date'] == forecast_date]
        if not row_val.empty:
            exog_dyn = row_val.iloc[0][exog_features].astype(float).values
            time_dyn = row_val.iloc[0]['time_index']
            exog = np.concatenate([exog_dyn, [time_dyn, np.sin(2 * np.pi * time_dyn / 12), np.cos(2 * np.pi * time_dyn / 12)]])
        else:
            exog = np.concatenate([anchor_exog, [forecast_time_index, sin_time, cos_time]])
        
        lag_feats = np.array([last_obs['Terakhir'], last_obs['Tertinggi'], last_obs['Terendah']])
        input_vector = np.concatenate([exog, lag_feats]).reshape(1, -1).astype(float)
        input_scaled = scaler.transform(input_vector)
        raw_pred = model.predict(input_scaled)[0]
    
    # Here, update state fully (you may add damping if desired).
    pred = raw_pred
    forecast_results.append({
        'date': forecast_date,
        'Forecasted Terakhir': pred[0],
        'Forecasted Tertinggi': pred[1],
        'Forecasted Terendah': pred[2]
    })
    last_obs['Terakhir'] = pred[0]
    last_obs['Tertinggi'] = pred[1]
    last_obs['Terendah'] = pred[2]

forecast_df = pd.DataFrame(forecast_results)

# Prepend the last historical observation so that the forecast line connects.
last_hist_df = pd.DataFrame({
    'date': [train_test['date'].max()],
    'Forecasted Terakhir': [train_test.iloc[-1]['Terakhir']],
    'Forecasted Tertinggi': [train_test.iloc[-1]['Tertinggi']],
    'Forecasted Terendah': [train_test.iloc[-1]['Terendah']]
})
forecast_df = pd.concat([last_hist_df, forecast_df], ignore_index=True)

forecast_df[['Forecasted Terakhir', 'Forecasted Tertinggi', 'Forecasted Terendah']] /= 1e5
train_test[targets] /= 1e5
validation[targets] /= 1e5

# --------------------------
# Accuracy Metrics
# --------------------------
# Define a function for Mean Absolute Percentage Error (MAPE).
def mape(y_true, y_pred):
    # To avoid division by zero, add a small epsilon.
    epsilon = 1e-6
    return np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100

# Compute accuracy on the Test set.
y_test_pred = model.predict(X_test_scaled) / 1e5
test_MAPE = {}
test_accuracy = {}
for i, t in enumerate(targets):
    mape_val = mape(y_test[t] / 1e5, y_test_pred[:, i])
    test_MAPE[t] = mape_val
    test_accuracy[t] = 100 - mape_val  # Accuracy as (100 - MAPE)
print("Test MAPE (%):", test_MAPE)
print("Test Accuracy (%):", test_accuracy)

# Compute accuracy on the Validation set (if forecast dates match validation dates).
val_df = validation.copy()
val_df[targets] /= 1e5
val_forecast = forecast_df[forecast_df['date'].isin(val_df['date'])]
if not val_forecast.empty:
    val_merge = pd.merge(val_forecast, val_df[['date'] + targets], on='date', how='left')
    val_MAPE = {}
    val_accuracy = {}
    for t in targets:
        mape_val = mape(val_merge[f'Forecasted {t}'], val_merge[t])
        val_MAPE[t] = mape_val
        val_accuracy[t] = 100 - mape_val
    print("Validation Accuracy (%):", val_MAPE)
else:
    print("No matching forecast dates in validation for accuracy evaluation.")

# --------------------------
# Visualization
# --------------------------
pd.options.display.float_format = '{:.8f}'.format
historical = pd.concat([train_test, validation]).sort_values('date')
colors = {'Terakhir': 'blue', 'Tertinggi': 'green', 'Terendah': 'red'}

fig = make_subplots(rows=1, cols=1)
for t in targets:
    fig.add_trace(
        go.Scatter(
            x=historical['date'],
            y=historical[t],
            mode='lines',
            name=f'Historical {t}',
            line=dict(color=colors[t], dash='solid')
        )
    )
for col, t in zip(['Forecasted Terakhir', 'Forecasted Tertinggi', 'Forecasted Terendah'], targets):
    fig.add_trace(
        go.Scatter(
            x=forecast_df['date'],
            y=forecast_df[col],
            mode='lines',
            name=f'Forecasted {t}',
            line=dict(color=colors[t], dash='solid')
        )
    )
fig.update_layout(
    title="Historical vs Forecasted Currency Variables (2 Years Ahead)",
    xaxis_title="Date",
    yaxis_title="Value",
    hovermode='x unified',
    xaxis=dict(type='date')
)
fig.show()

table_fig = go.Figure(data=[go.Table(
    header=dict(values=list(forecast_df.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[forecast_df[col] for col in forecast_df.columns],
               fill_color='lavender',
               align='left'))
])
table_fig.update_layout(title="Forecasted Values Table (2 Years Ahead)")
table_fig.show()

# Forecast vs Actual (Validation) Table.
val_forecast = forecast_df[forecast_df['date'].isin(val_df['date'])]
results_table = pd.merge(
    val_forecast,
    val_df[['date'] + targets].rename(columns={
        'Terakhir': 'Actual Terakhir',
        'Tertinggi': 'Actual Tertinggi',
        'Terendah': 'Actual Terendah'
    }),
    on='date', how='left'
)
print("Forecast vs Actual (Validation) Table:")
print(results_table)


Test MSE: 25934591.365977645
Test MAPE (%): {'Terakhir': 6.026928282709225, 'Tertinggi': 4.554201801348945, 'Terendah': 5.771239840021725}
Test Accuracy (%): {'Terakhir': 93.97307171729078, 'Tertinggi': 95.44579819865106, 'Terendah': 94.22876015997828}
Validation Accuracy (%): {'Terakhir': 99.99894029924226, 'Tertinggi': 99.99903756812776, 'Terendah': 99.99901907002788}


/opt/anaconda3/lib/python3.11/site-packages/sklearn/base.py:439: UserWarning:

X does not have valid feature names, but StandardScaler was fitted with feature names

/opt/anaconda3/lib/python3.11/site-packages/sklearn/base.py:439: UserWarning:

X does not have valid feature names, but StandardScaler was fitted with feature names

/opt/anaconda3/lib/python3.11/site-packages/sklearn/base.py:439: UserWarning:

X does not have valid feature names, but StandardScaler was fitted with feature names

/opt/anaconda3/lib/python3.11/site-packages/sklearn/base.py:439: UserWarning:

X does not have valid feature names, but StandardScaler was fitted with feature names

/opt/anaconda3/lib/python3.11/site-packages/sklearn/base.py:439: UserWarning:

X does not have valid feature names, but StandardScaler was fitted with feature names

/opt/anaconda3/lib/python3.11/site-packages/sklearn/base.py:439: UserWarning:

X does not have valid feature names, but StandardScaler was fitted with feature names

/opt

Forecast vs Actual (Validation) Table:
        date  Forecasted Terakhir  Forecasted Tertinggi  Forecasted Terendah  \
0 2024-11-30           0.99931539            1.12523297           1.05787602   
1 2024-12-31           1.00224617            1.12750797           1.05827930   

   Actual Terakhir  Actual Tertinggi  Actual Terendah  
0       0.00000945        0.00000989       0.00000943  
1       0.00000976        0.00000979       0.00000932  
